# Filter LSST alerts to select high redshift objects

[Getting LSST alerts with the Babamul API](https://github.com/boom-astro/babamul/blob/main/examples/stream-basic/notebook.ipynb)

### Notes on running this notebook:

This notebook fetches alerts using the Babamul broker, which requires [user credentials saved in a .env file](https://github.com/boom-astro/babamul/blob/main/examples/stream-basic/.env.example). 

Babamul has periodically changed the naming schema, for example the token was previously changed from BABAMUL_API_TOKEN to BABAMUL_TOKEN, so the .env.example linked here may be incorrect.

In [ ]:
#imports 
import dotenv
dotenv.load_dotenv()

### Catalog uniformization

The following code may need to be edited to work for each specific catalog.

We cut the full catalog to subsection of confident galaxies

We rename columns with standardized schema (id, ra, dec, z, z unc, and any unique values that we want to keep for a given catalog. If possible save the magnitude of the galaxy as host_mag)

- z_unc saves the 1 sigma uncertainty of the redshift.

- If possible we save ra/dec uncertainties, but if they only exist as global values they are recorded in catalogs_catalog.

In [ ]:
from astropy.table import Table, hstack                                                                                                                  
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt

This code processes the Astrodeep catalogs from JWST

Note on Astrodeep: No per-source astrometric errors are provided in the ASTRODEEP-JWST catalog (Merlin et al. 2024). Images are aligned to Gaia-DR3; we adopt a conservative positional uncertainty of 0.5 arcsec for crossmatching.

In [ ]:
# we load the full downloaded catalog

with fits.open('../../data/catalogs/raw_catalogs/ASTRODEEP-JWST_photoz/ABELL2744_photoz.fits') as hdul:
    hdul.info()

t = Table.read('../../data/catalogs/raw_catalogs/ASTRODEEP-JWST_photoz/ABELL2744_photoz.fits')                                                                                                 
t[0:5]

In [ ]:
# inspect catalog contents, ie histogram redshift values

import matplotlib.pyplot as plt

plot = plt.hist(t['zspec'], bins=50, range=(0,6))

In [ ]:
# here are our cuts for the astrodeep catalogs (http://www.astrodeep.eu/astrodeep-jwst-catalogs/)
# see Merlin 2024 for description of data, flags etc (https://arxiv.org/abs/2409.00169)

galaxies = t[
    (t['flag'] < 100000) # spurious sources
]

print(f'selected {len(galaxies)} from {len(t)} total objects')

# and then here is where we standardize column names from the astrodeep names
# we dont cut any columns as astrodeep is already very concise.

cut_catalog = galaxies.copy()
cut_catalog.rename_column('ID', 'id')
cut_catalog.rename_column('RA', 'ra')
cut_catalog.rename_column('DEC', 'dec')
cut_catalog['z'] = np.where(cut_catalog['zspec'] != -99.0, cut_catalog['zspec'], cut_catalog['zphot']) # use zspec where possible  

# z uncertainty                         
# spec-z: use global assumption σ_z/(1+z) = 0.001 (typical MOS precision;
#         paper validates spec-z quality via flag filtering but gives no explicit value)                                                                                    
# photo-z: per-object std of the 4 EAzY estimates, floored at the global NMAD
#          NMAD = 0.031 from Merlin+2024 (Table in paper)                                                                                                                   
                                                                                                                                                                            
_SPECZ_SIGMA_FRAC = 0.001   # σ_z/(1+z) for spec-z                                                                                                                          
_PHOTOZ_NMAD      = 0.031   # global photo-z NMAD from paper                                                                                                                
                                                                                                                                                                            
_photoz_cols = ['EAzY_eazy_v13', 'EAzY_Larson', 'EAzY_LarsonLyaRed', 'zphot']                                                                                               
_photoz_stack = np.column_stack([cut_catalog[c] for c in _photoz_cols]).astype(float)                                                                                       
_photoz_per_obj_std = np.std(_photoz_stack, axis=1)                                                                                                                         
_photoz_nmad_floor  = _PHOTOZ_NMAD * (1 + cut_catalog['z'])                                                                                                                 
                                            
_has_specz = cut_catalog['zspec'] != -99.0                                                                                                                                  
cut_catalog['z_unc'] = np.where(                                                                                                                                          
    _has_specz,                                                                                                                                                             
    _SPECZ_SIGMA_FRAC * (1 + cut_catalog['z']),                                                                                                                           
    np.maximum(_photoz_per_obj_std, _photoz_nmad_floor),                                                                                                                    
) 

In [ ]:
# and then we print some statistics on the selected objects, which are copied into catalogs_catalog.py

# Sky coverage (assuming contiguous rectangular field)
ra_min, ra_max = cut_catalog['ra'].min(), cut_catalog['ra'].max()
dec_min, dec_max = cut_catalog['dec'].min(), cut_catalog['dec'].max()
dec_mean_rad = np.radians((dec_min + dec_max) / 2)
area_sqdeg = (ra_max - ra_min) * np.cos(dec_mean_rad) * (dec_max - dec_min)
print(f'Approximate sky coverage: {area_sqdeg:.2f} deg^2')

# object stats
for col in ['ra', 'dec', 'z']:                                                                                                                                         
      print(f"{col}: min={cut_catalog[col].min():.4f}, max={cut_catalog[col].max():.4f}") 

print(f'median z: {np.median(cut_catalog['z']):.4f}')

# finally we save the cut catalog to a fits file
cut_catalog.write('../../data/catalogs/ASTRODEEP_ABELL2744_cut.fits', overwrite=False) 

In [ ]:
from thor.utils.rubin_stats_visualizations import plot_skymap

plot_skymap(
    alerts=None,
    plot_catalogs=True,
    plot_ddf=False,
    catalog_path="../../data/catalogs",
    title="Catalog Coverage Skymap",
)

This code processes the Cosmos2025 catalog from JWST

In [ ]:
# we load the full downloaded catalog. 
# combine 1: Photometry (PSF-homogenized Aperture and SE++ Model-Based) and 2: LePhare Photometric Redshifts and Physical Parameters

with fits.open('../../data/catalogs/COSMOSWeb_mastercatalog_v1.1_photom_primary.fits') as hdul:
    hdul.info()

with fits.open('../../data/catalogs/COSMOSWeb_mastercatalog_v1.1_lephare.fits') as hdul:
    hdul.info()
                                                                                                                                                        
t_photom = Table.read('../../data/catalogs/COSMOSWeb_mastercatalog_v1.1_photom_primary.fits')                                                            
t_lephare = Table.read('../../data/catalogs/COSMOSWeb_mastercatalog_v1.1_lephare.fits')                                                                
                                                                                                                                                        
# Both tables have 784016 rows in the same order — row-matched, so hstack directly                                                                       
t = hstack([t_photom, t_lephare])                                                                                                                      
t[0:5]                                                                                                                                                   
        

In [ ]:
# inspect catalog contents, ie histogram redshift values

import matplotlib.pyplot as plt

plot = plt.hist(t['zfinal'], bins=50, range=(0,6))

In [ ]:
# values to select on are specific to the catalog                                                                                                        
                                                                                                                                                        
# cuts for COSMOSWeb                                                                                  
# lephare type: 0=galaxy, 1=star, 2=AGN                                                                                                                  
galaxies = t[                                                                                                                                            
    (t['type'] == 0) &          # LePhare classified as galaxy                                                                                           
    (t['zfinal'] > 0) &         # has a valid redshift                                                                                                   
    (t['flag_star'] == False) &  # not a morphological star (SE++ flag)                                                                                  
    (t['flag_star_hsc'] == 0) &  # not a star per HSC (0=galaxy, 1=star)                                                                                 
    (t['flag_blend'] == False) & # not a blended source                                                                                                  
    (t['warn_flag'] == 0)        # no photometry/fit warnings                                                                                            
]                                                                                                                                                        
                                                                                                                                                        
print(f'selected {len(galaxies)} from {len(t)} total objects')                                                                                           
                                                                                                                                                        
# standardize column names                                                                                                                               
cut_catalog = galaxies['id', 'ra', 'dec', 'zfinal', 'zpdf_l68', 'zpdf_u68',                                                                              
                          'chi2_best', 'type', 'flag_chandra',                                                                                           
                          'mabs_nuv', 'mabs_r', 'mabs_k', 'mass_med'].copy()                                                                                    
                                                                                                                                                        
cut_catalog.rename_column('zfinal', 'z')                                                                                                                 
                                                                                                                                                        
# z_unc saved as 1 sigma uncertainty from 68% confidence interval half-width                                                                                                          
cut_catalog['z_unc'] = (cut_catalog['zpdf_u68'] - cut_catalog['zpdf_l68']) / 2 

In [ ]:
# and then we print some statistics on the selected objects, which are copied into catalogs_catalog.py

# Sky coverage (assuming contiguous rectangular field)
ra_min, ra_max = cut_catalog['ra'].min(), cut_catalog['ra'].max()
dec_min, dec_max = cut_catalog['dec'].min(), cut_catalog['dec'].max()
dec_mean_rad = np.radians((dec_min + dec_max) / 2)
area_sqdeg = (ra_max - ra_min) * np.cos(dec_mean_rad) * (dec_max - dec_min)
print(f'Approximate sky coverage: {area_sqdeg:.2f} deg^2')

# object stats
for col in ['ra', 'dec', 'z']:                                                                                                                                         
      print(f"{col}: min={cut_catalog[col].min():.4f}, max={cut_catalog[col].max():.4f}") 

print(f'median z: {np.median(cut_catalog['z']):.4f}')

# finally we save the cut catalog to a fits file
cut_catalog.write('../../data/catalogs/COSMOS2025_cut.fits', overwrite=False) 

In [ ]:
from thor.utils.rubin_stats_visualizations import plot_skymap

plot_skymap(
    alerts=None,
    plot_catalogs=True,
    plot_ddf=False,
    catalog_path="../../data/catalogs",
    title="Catalog Coverage Skymap",
)

This code processes the CLAUDS catalogs from Subaru and GHST
- separate catalogs for Cosmos, Deep23, and XMMLss fields, fourth field is northern hemisphere
- They test HSCpipe as well as the commonly used Source Extractor+LePhare pipeline, we use the latter.

In [ ]:
# we load the full downloaded catalog
# download XMM-LSS photometry, DEEP2-3 photometry, E-COSMOS photometry

with fits.open('../../data/catalogs/raw_catalogs/XMMLSS_11bands-SExtractor-Lephare.fits') as hdul:
    hdul.info()

t = Table.read('../../data/catalogs/raw_catalogs/XMMLSS_11bands-SExtractor-Lephare.fits')                                                                                                 
t[0:5]

In [ ]:
# inspect catalog contents, ie histogram redshift values

import matplotlib.pyplot as plt

plot = plt.hist(t['ZPHOT'], bins=50, range=(0,6))

In [ ]:
# values to select on are specific to the catalog                                                                                                        
                                                                                                                                                        
# cuts for CLAUDS                                                                                
# lephare type: 0=galaxy, 1=star, 2=AGN                                                                                                                  
galaxies = t[
      (t['MASK'] == 0) &          # not in bright star mask
      (t['ST_TRAIL'] == 0) &       # not in satellite trail
      (t['OBJ_TYPE'] == 0) &       # galaxy (not star or QSO)
      (t['Z_BEST'] > 0) &          # valid photo-z
      (t['Z_BEST'] < 6) &
      (t['i'] < 25.0)              # reliable photo-z depth
]                    
                                                                                                                                                        
print(f'selected {len(galaxies)} from {len(t)} total objects')  

magerr = np.clip(galaxies['MAGERR_APER_2s_i'], 1e-6, None)                                                                                                                  
snr = 1.0857 / magerr                                                                                                                           
# RMS source size from ellipse axes (degrees -> arcsec)                                                                                                                     
sigma_ab_arcsec = np.sqrt(galaxies['A_WORLD'] * galaxies['B_WORLD']) * 3600                                                                                                                                                                 
# Statistical centroid uncertainty                                                                                                                                        
sigma_stat = sigma_ab_arcsec / snr                                                                                                                                                                                                                                                                                                            
sigma_sys = 0.010  # HSC astrometric systematic floor ~10 mas (Aihara+2018)                                                                                                                                                                                                                                                                                                                 
galaxies['pos_unc'] = np.sqrt(sigma_stat**2 + sigma_sys**2)

# z_unc saved as 1 sigma uncertainty from 68% confidence interval half-width                                                                                                          
galaxies['z_unc'] = (galaxies['Z_BEST68_HIGH'] - galaxies['Z_BEST68_LOW']) / 2 
                                                                                                                                                        
# standardize column names                                                                                                                               
cut_catalog = galaxies['ID', 'RA', 'DEC', 'Z_BEST', 'z_unc', 'pos_unc', 'Z_BEST68_LOW', 'Z_BEST68_HIGH',                                                                                              
                        'CHI_BEST', 'OBJ_TYPE', 'CLASS_STAR_HSC_I', 'i', 'i_err', 'MAG_APER_2s_i', 'MAGERR_APER_2s_i'].copy()                                                                                    

cut_catalog.rename_column('ID', 'id')                                                                                                                                     
cut_catalog.rename_column('RA', 'ra')
cut_catalog.rename_column('DEC', 'dec')
cut_catalog.rename_column('Z_BEST', 'z')                                                                                                                                                                                                                                                

In [ ]:
# and then we print some statistics on the selected objects, which are copied into catalogs_catalog.py

# Sky coverage (assuming contiguous rectangular field)
ra_min, ra_max = cut_catalog['ra'].min(), cut_catalog['ra'].max()
dec_min, dec_max = cut_catalog['dec'].min(), cut_catalog['dec'].max()
dec_mean_rad = np.radians((dec_min + dec_max) / 2)
area_sqdeg = (ra_max - ra_min) * np.cos(dec_mean_rad) * (dec_max - dec_min)
print(f'Approximate sky coverage: {area_sqdeg:.2f} deg^2')

# object stats
for col in ['ra', 'dec', 'z']:                                                                                                                                         
      print(f"{col}: min={cut_catalog[col].min():.4f}, max={cut_catalog[col].max():.4f}") 

print(f'median z: {np.median(cut_catalog['z']):.4f}')

# finally we save the cut catalog to a fits file
cut_catalog.write('../../data/catalogs/CLAUDS_SourceExtractor_XMMLSS_cut.fits', overwrite=False) 

In [ ]:
from thor.utils.rubin_stats_visualizations import plot_skymap

plot_skymap(
    alerts=None,
    plot_catalogs=True,
    plot_ddf=False,
    catalog_path="../../data/catalogs",
    title="Catalog Coverage Skymap",
)

### Fetching, prefiltering, organizing LSST alerts

Develop pipeline to crossmatch (or do any inference) on sources locally

In [ ]:
# get alerts from date range and other basic filters for astrophysical sources
# this has crashed for large date ranges, so is built to save alerts in batches.

from thor.utils.fetch_alerts import babamul_get_alerts

start="07-01-2026"
end="07-10-2026"

loaded_alerts = babamul_get_alerts(
    survey="LSST",
    start_time=start,
    end_time=end,
    min_drb=0.4,
    is_rock=False,
    is_star=False,
    is_near_brightstar=False,
    is_stationary=True,
) 

In [ ]:
# Bookeeping: if we saved batches as we fetched, combine the batched files of alerts into a single compressed file and move out of raw_files
from thor.utils.fetch_alerts import combine_alert_files
from datetime import datetime

start_fmt = datetime.strptime(start, "%m-%d-%Y").strftime("%m%d%y")                                                                                                         
end_fmt = datetime.strptime(end, "%m-%d-%Y").strftime("%m%d%y")
filename = f"basic_{start_fmt}_{end_fmt}.json.gz" 

combined = combine_alert_files(input_dir="../../data/lsst_alert_download/raw_files/", output_path=f"../../data/lsst_alert_download/{filename}", delete_raw=True)

In [ ]:
# Bookkeeping: combine saved alerts to bigger batches per file
# this will deduplicated using objectId when running

from thor.utils.fetch_alerts import combine_alert_files

combined = combine_alert_files(                                                                                                                                           
    input_dir="../../data/lsst_alert_download/",                                                                                                                      
    input_files=["basic_041226_062926.json.gz", "basic_062626_070226.json.gz"],
    output_path="../../data/lsst_alert_download/basic_041226_070226.json.gz",
    delete_raw=True                                                                                                         
)

In [ ]:
# if we want to directly open a locally saved alerts file, we can do that too

from thor.utils.fetch_alerts import load_alerts
loaded_alerts = load_alerts("../../data/lsst_alert_download/basic_010126_041426.json.gz")

In [ ]:
# additional filtering

# in generic_filter we make some more generic cuts to get real astrophysical objects
# filter out dipole (image subtraction artifacts), poor PSF fits, low SNR, extended sources, and alerts with shape or centroid fitting issues. 
# filter currently repeats cuts made in babamul_get_alerts: drb<0.4, rock, star, near_brightstar, isdiffpos, and stationary.

from thor.utils.filter_functions import (filter_alerts, generic_filter, deduplicate_alerts)

filtered_alerts = filter_alerts(                                                                                                           
    loaded_alerts,                                                            
    generic_filter,                                                                                                                        
) 

# deduplicate alerts to get unique objects
filtered_objects = deduplicate_alerts(filtered_alerts)

### Catalog crossmatch

In [ ]:
import importlib
import thor.utils.fetch_alerts
import thor.utils.filter_functions

importlib.reload(thor.utils.fetch_alerts)
importlib.reload(thor.utils.filter_functions)

In [ ]:
# crossmatch with all available catalogs, ie all .fits files in data/catalogs
# returns a dictionary with all matched alert and catalog info.
# can provide catalog_name as a string or list of strings to specify which catalogs in catalog_path (default data/catalogs) to use. 
# user must download catalogs and ensure there are columns with names id, ra, dec, and z - the function is not flexible in 
# identifying different naming schema for these columns.

from thor.utils.filter_functions import catalog_crossmatch

crossmatched_objects = catalog_crossmatch(
      alerts=filtered_objects,
)

In [ ]:
# additional filtering on catalog features

from thor.utils.filter_functions import catalog_filter

filtered = catalog_filter(
    crossmatched_objects,
    z_min = 0.5,
)

In [ ]:
# Source specific filtering:
# in tde_filter we also make more specific cuts, including minimum # detections, no milliquas matches

from thor.utils.filter_functions import (filter_alerts, tde_filter)

filtered = filter_alerts(                                                                                                           
    filtered,                                                            
    tde_filter
) 

### loading and inspecting objects

In [ ]:
crossmatched_alerts = [v["LSST"] for v in crossmatched_objects.values()]
print(f"Total crossmatched alerts: {len(crossmatched_alerts)}")

filtered_crossmatched_alerts = [v["LSST"] for v in filtered.values()]  
print(f"Filtered crossmatched alerts: {len(filtered_crossmatched_alerts)}")

In [ ]:
# inspect all crossmatch results without the cuts on rise etc.

from babamul.jupyter import scan_alerts
from astropy.time import Time
print(f'Todays JD: {round(Time.now().jd)}')
scan_alerts(filtered_crossmatched_alerts)

In [ ]:
# copy lsst id to print crossmatch info for a given id from dictionary

id = '313967767357227074'

obj = crossmatched_objects[id]
for catalog, data in obj.items():
     if catalog != "LSST" and data is not None:
         print(f"{catalog}:")
         print(f"end=z = {data['z']:.2f}") 
         if "CHI_BEST" in data:
            print(f"chi2 = {data['CHI_BEST']:.2f}")
         if "consearch_arcsecs" in data:
            print(f"separation = {data['consearch_arcsecs']:.2f} arcsec")


In [ ]:
# check abs mag given apparent mag and redshift
m=22
z=2.1

from astropy.cosmology import Planck18 as cosmo                                                                                                                   
abs_mag = m - cosmo.distmod(z).value
print(f"Apparent magnitude: {m}, redshift: {z}, absolute magnitude: {abs_mag:.2f}")

In [ ]:
# by hand create a list of objects that pass inspection, save
# will add objectid to existing file, without duplicating objects

cand_ids_to_save = []

from thor.utils.fetch_alerts import save_objects

save_objects(cand_ids_to_save, path="../../data/lsst_alert_download/tde_candidates.json.gz")

### additional ways to fetch objects

In [ ]:
# or we provide function to run cosmos crossmatch by directly provided ra and dec instead of alert
# you may provide a single ra and dec, or a list of ras and a list of corresponding decs as arguments.

from thor.utils.filter_functions import catalog_crossmatch

crossmatched_coords = catalog_crossmatch(
    ra=150.101273,
    dec=2.743894,
    radius_arcsec=300,
    catalog_path="../../data/catalogs",
)

In [ ]:
# or we can load our list of previously saved candidates, and fetch alerts to display

from thor.utils.filter_functions import catalog_crossmatch
from thor.utils.fetch_alerts import (load_objects, fetch_latest_alerts)

object_ids = load_objects("../../data/lsst_alert_download/tde_candidates.json.gz")
latest_alerts = fetch_latest_alerts(object_ids, save=False, verbose=False)
saved_objects = catalog_crossmatch(latest_alerts)

In [ ]:
# look at our candidates:

from babamul.jupyter import scan_alerts
from astropy.time import Time
print(f'Todays JD: {round(Time.now().jd)}')

show_saved_objects = [v["LSST"] for v in saved_objects.values()]  
scan_alerts(show_saved_objects)

In [ ]:
# copy lsst id to print crossmatch info for a given id from dictionary

id = '170591539376422992'

obj = saved_objects[id]
for catalog, data in obj.items():
     if catalog != "LSST" and data is not None:
         print(f"{catalog}:")
         print(f"end=z = {data['z']:.2f}") 
         if "CHI_BEST" in data:
            print(f"chi2 = {data['CHI_BEST']:.2f}")
         if "consearch_arcsecs" in data:
            print(f"separation = {data['consearch_arcsecs']:.2f} arcsec")


### prost - currently developing bayesian crossmatch instead of basic conesearch

https://github.com/alexandergagliano/Prost

notes: set priors on z, offset, host mag.

missed catalog probability: true host too faint and undetected in image

small cone probability: search radius too small

can return top 2 best host associations.

In [ ]:
import importlib
import thor.utils.fetch_alerts
importlib.reload(thor.utils.fetch_alerts)

In [ ]:
# test Prost out of the box / basic usage

import pandas as pd
from astro_prost import associate_sample
from scipy.stats import gamma, halfnorm, uniform

# define a transient catalog 
transient_catalog = pd.DataFrame({
    'name': ['MyTransient'],
    'ra': [350.838161],
    'dec': [0.101394]
})

# define a set of catalogs to search -- options are glade, decals, panstarrs, and skymapper
catalogs = ["decals", "glade"]

# define priors and likelihoods
priorfunc_offset = uniform(loc=0, scale=10)
likefunc_offset = gamma(a=0.75)

priors = {"offset": priorfunc_offset}
likes = {"offset": likefunc_offset}

# associate
hosts = \
    associate_sample(
        transient_catalog,
        priors=priors,
        likes=likes,
        catalogs=catalogs,
        name_col='name',
        coord_cols=('ra', 'dec'),
        save=False
)

In [ ]:
# test prost with custom patch to handle unsuported catalogs

from pathlib import Path    
import pandas as pd                                                                                                                                                
from astropy.table import Table                                                                                                                                             
from thor.utils.prost_catalogs import register_local_catalog
from astro_prost import associate_sample
from scipy.stats import gamma, halfnorm, uniform


# custom local catalogs                                                                                                              
CATALOG_DIR = Path.cwd().parents[1] / "data" / "catalogs"
df = Table.read(CATALOG_DIR / "COSMOS2025_cut.fits").to_pandas()                                                                                                            
register_local_catalog("cosmos2025", df)

# define a transient catalog 
transient_catalog = pd.DataFrame({
    'name': ['MyTransient'],
    'ra': [150.1],
    'dec': [2.2]
})

# define a set of catalogs to search -- options are glade, decals, panstarrs, and skymapper
catalogs = ["cosmos2025"]

# define priors and likelihoods
priorfunc_offset = uniform(loc=0, scale=10)
likefunc_offset = gamma(a=0.75)
priors = {"offset": priorfunc_offset}
likes = {"offset": likefunc_offset}

# associate
hosts = \
    associate_sample(
        transient_catalog,
        priors=priors,
        likes=likes,
        catalogs=catalogs,
        name_col='name',
        coord_cols=('ra', 'dec'),
        save=False
)

### get distance in parsecs from angular separation

In [ ]:
import numpy as np
from astropy.cosmology import Planck18 as cosmo                                                                                                               
import astropy.units as u

In [ ]:
# pick any matched object
from thor.utils.filter_functions import catalog_crossmatch                                                                                                       
from thor.utils.fetch_alerts import load_objects, fetch_latest_alerts                                                                                            
                                                                                                                                                                
object_ids = load_objects("../../data/lsst_alert_download/tde_candidates.json.gz")                                                                             
latest_alerts = fetch_latest_alerts(object_ids, save=False, verbose=False)                                                                                       
saved_objects = catalog_crossmatch(latest_alerts)                                                                                                              
                                                                                                                                                                
# pick object with a COSMOS2025 match
obj_id = next(                                                                                                                                                   
    oid for oid, obj in saved_objects.items()                                                                                                                  
    if obj.get('CLAUDS_SourceExtractor_COSMOS_cut') is not None
)                                                                                                                                                                
obj = saved_objects[obj_id]
                                                                                                                                                                
alert = obj['LSST']                                                                                                                                            
catalog_entry = obj['CLAUDS_SourceExtractor_COSMOS_cut']

print(f"z={catalog_entry.get('z')}, sep={catalog_entry.get('conesearch_arcsecs'):.2f}\"")                                                                        
print(f"ra={catalog_entry.get('ra')}, dec={catalog_entry.get('dec')}")

In [ ]:
import numpy as np
from astropy.cosmology import Planck18 as cosmo
import astropy.units as u
from astropy.coordinates import SkyCoord

def angular_sep_to_parsecs_with_error(redshift, separation_arcsec, sigma_sep_arcsec, sigma_z):
    """
    Convert angular separation to physical separation in parsecs,
    with error propagation from position and redshift uncertainties.

    Uses angular diameter distance d_A = d_C / (1+z), which correctly
    accounts for photons traveling through expanding space — objects
    appear larger on the sky than the naive Euclidean distance would suggest.

    Total uncertainty from two independent sources:
        σ_d² = (d_A × σ_sep_rad)²  +  (dd_A/dz × sep_rad × σ_z)²

    Parameters
    ----------
    redshift : float
    separation_arcsec : float
    sigma_sep_arcsec : float
        1-sigma uncertainty on the angular separation in arcsec.
        Combine alert and catalog astrometric errors in quadrature before passing.
    sigma_z : float
        1-sigma redshift uncertainty.
        Spectroscopic: ~0.001 (negligible). Photometric: ~0.03–0.05 (can dominate).

    Returns
    -------
    sep_pc : float
        Physical separation in parsecs.
    sep_err_pc : float
        1-sigma uncertainty in parsecs.
    """
    arcsec_to_rad = (1 * u.arcsec).to(u.rad).value

    # angular diameter distance (not luminosity distance)
    d_A_pc = cosmo.angular_diameter_distance(redshift).to(u.pc).value

    # physical separation
    sep_rad = separation_arcsec * arcsec_to_rad
    sep_pc = d_A_pc * sep_rad

    # position term: uncertainty in the angular separation
    sigma_sep_rad = sigma_sep_arcsec * arcsec_to_rad
    err_position = d_A_pc * sigma_sep_rad

    # redshift term: dd_A/dz computed numerically
    dz = 1e-4
    d_A_plus  = cosmo.angular_diameter_distance(redshift + dz).to(u.pc).value
    d_A_minus = cosmo.angular_diameter_distance(redshift - dz).to(u.pc).value
    dd_A_dz = (d_A_plus - d_A_minus) / (2 * dz)
    err_redshift = dd_A_dz * sep_rad * sigma_z

    # total error (position and redshift uncertainties are independent)
    sep_err_pc = np.sqrt(err_position**2 + err_redshift**2)

    return sep_pc, sep_err_pc

In [ ]:
from thor.catalogs_catalog import catalogs as _CATALOGS_META

def _catalog_pos_unc_arcsec(catalog_name: str) -> tuple[float, float]:
    """
    Return (ra_unc_arcsec, dec_unc_arcsec) for a catalog, sourced from catalogs_catalog.py.

    Looks up global_ra_unc_mas / global_dec_unc_mas from the catalog metadata.
    Falls back to (0.0, 0.0) if not present or None.
    """
    # catalog_name may be a stem (no .fits) or a full filename
    key = catalog_name if catalog_name.endswith('.fits') else catalog_name + '.fits'
    entry = _CATALOGS_META.get(key, {})
    ra_mas  = entry.get("global_ra_unc_mas")
    dec_mas = entry.get("global_dec_unc_mas")
    ra_unc_arcsec  = (ra_mas  / 1000.0) if ra_mas  is not None else 0.0
    dec_unc_arcsec = (dec_mas / 1000.0) if dec_mas is not None else 0.0
    return ra_unc_arcsec, dec_unc_arcsec


def get_lsst_catalog_separation_parsecs(alert, catalog_entry, catalog_name: str):
    """
    Compute physical separation (pc) between an alert and a catalog object,
    with full error propagation.

    Position uncertainty priority:
      1. Per-source: catalog_entry['pos_unc'] (arcsec) if present and not None.
         Treated as a symmetric 1-sigma positional uncertainty — used directly
         as catalog_sigma_arcsec (NOT as sep_arcsec).
      2. Global: global_ra_unc_mas / global_dec_unc_mas from catalogs_catalog.py,
         combined in quadrature as sqrt(ra_unc^2 + dec_unc^2).
      3. Fallback: 0.05 arcsec (50 mas) if neither is available.

    The angular separation (sep_arcsec) is always computed from SkyCoord.
    """
    # --- alert position ---
    if hasattr(alert, 'candidate'):
        alert_ra  = alert.candidate.ra
        alert_dec = alert.candidate.dec
        alert_ra_unc_deg  = alert.candidate.raErr  or 0.0
        alert_dec_unc_deg = alert.candidate.decErr or 0.0
    else:
        alert_ra  = alert['ra']
        alert_dec = alert['dec']
        alert_ra_unc_deg  = alert.get('raErr')  or 0.0
        alert_dec_unc_deg = alert.get('decErr') or 0.0

    catalog_ra        = catalog_entry['ra']
    catalog_dec       = catalog_entry['dec']
    catalog_redshift  = catalog_entry['z']
    catalog_redshift_unc = catalog_entry['z_unc']

    # --- angular separation (always from SkyCoord) ---
    c1 = SkyCoord(ra=alert_ra * u.deg, dec=alert_dec * u.deg)
    c2 = SkyCoord(ra=catalog_ra * u.deg, dec=catalog_dec * u.deg)
    sep_arcsec = c1.separation(c2).to(u.arcsec).value

    # --- catalog position uncertainty ---
    pos_unc = catalog_entry.get('pos_unc') if isinstance(catalog_entry, dict) else None
    if pos_unc is not None:
        # per-source uncertainty in arcsec, symmetric
        catalog_sigma_arcsec = float(pos_unc)
    else:
        ra_unc_arcsec, dec_unc_arcsec = _catalog_pos_unc_arcsec(catalog_name)
        if ra_unc_arcsec > 0.0 or dec_unc_arcsec > 0.0:
            catalog_sigma_arcsec = np.sqrt(ra_unc_arcsec**2 + dec_unc_arcsec**2)
        else:
            print(f"Warning: no positional uncertainty for '{catalog_name}', assuming 50 mas.")
            catalog_sigma_arcsec = 0.05  # 50 mas fallback

    # --- alert position uncertainty (deg -> arcsec, RA needs cos(dec) correction) ---
    alert_ra_unc_arcsec  = alert_ra_unc_deg  * 3600 * np.cos(np.radians(alert_dec))
    alert_dec_unc_arcsec = alert_dec_unc_deg * 3600

    # --- combined separation uncertainty (all terms independent) ---
    sigma_sep_arcsec = np.sqrt(
        alert_ra_unc_arcsec**2 + alert_dec_unc_arcsec**2
        + catalog_sigma_arcsec**2
    )

    sep_pc, sep_err_pc = angular_sep_to_parsecs_with_error(
        redshift=catalog_redshift,
        separation_arcsec=sep_arcsec,
        sigma_sep_arcsec=sigma_sep_arcsec,
        sigma_z=catalog_redshift_unc,
    )
    return sep_pc, sep_err_pc

In [ ]:
# for reference galaxies are ~1-100kpc in diameter (0.001-0.1Mpc)
# nuclear search radii depend on instrument resolution, but are typically tens of pc up to 1-2kpc   

sep_pc, sep_err_pc = get_lsst_catalog_separation_parsecs(alert, catalog_entry, catalog_name="CLAUDS_SourceExtractor_COSMOS_cut")

print(f"Physical separation: {sep_pc:.2f} pc ± {sep_err_pc:.2f} pc")

In [ ]:
import importlib
import thor.utils.fetch_alerts
importlib.reload(thor.utils.fetch_alerts)

### rubin nuclearity

In [ ]:
# given __ pc physical search radius for nuclear transients, with perfect systematics what arcsecond radius would we need to search at a given redshift?

# can calculate x pc from assuming avg (10^8 black hole), calculate sphere of influence

In [ ]:
# now check Rubin systematics: calculate dispersion of ra and dec positions for some transients
# this should be limiting factor

In [ ]:
# plot dispersion as a function of airmass

In [ ]:
# plot dispersion as a function of ccd position (closeness to center)